# Customer Lifetime Value Prediction with CRISP-DM

End-to-end regression project to estimate customer LTV at the time of the first purchase.

## 1. Business Understanding

**Goal:** estimate expected Customer Lifetime Value (LTV) using only information available at the first purchase, supporting acquisition and paid-media decisions.

**Success metrics:** MAE, RMSE, and R².

## 2. Data Understanding

The prepared dataset contains **38,753 customers**. The original educational dataset is not redistributed in this repository.

The source dataset uses Portuguese labels. The pipeline translates the source schema into English before modeling.

In [ ]:
import pandas as pd
import sys

DATA_PATH = "../data/ltv_base_tratada_cardinalidade_final.csv"
raw_df = pd.read_csv(DATA_PATH)

sys.path.append("../src")
from ltv_pipeline import translate_source_schema

df = translate_source_schema(raw_df)
df.shape
df.head()

## 3. Data Preparation

- Target: `LTV`
- `StandardScaler`: `first_purchase_value`
- `OneHotEncoder(drop="first")`: categorical features, including purchase month
- `recurring_first_purchase`: passed through unchanged
- 80/20 train-test split with `random_state=42`
- Preprocessing remains inside the pipeline to reduce leakage risk.

In [ ]:
from ltv_pipeline import build_preprocessor, split_data

X_train, X_test, y_train, y_test = split_data(df)
preprocessor = build_preprocessor()
X_train.shape, X_test.shape

## 4. Modeling

Four models are compared: DummyRegressor, Linear Regression, Poly (d=2), and RF.

Five-fold cross-validation uses **R²** on the training set.

In [ ]:
from ltv_pipeline import cross_validate_models

cv_results = cross_validate_models(X_train, y_train)
cv_results

### Cross-validation results observed in the project

| Model | Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 | Mean R² |
|---|---:|---:|---:|---:|---:|---:|
| Dummy | -0.000101 | -0.002614 | -0.001122 | -0.000027 | -0.000418 | -0.000856 |
| **Linear Regression** | **0.848888** | **0.843970** | **0.853254** | **0.848635** | **0.857156** | **0.850381** |
| Poly (d=2) | 0.843369 | 0.839878 | 0.850105 | 0.845545 | 0.853342 | 0.846448 |
| RF | 0.820993 | 0.815219 | 0.831260 | 0.825405 | 0.833917 | 0.825359 |

## 5. Evaluation

Linear Regression was selected because it combines the strongest cross-validation performance with interpretability.

**Test set:** R² = 0.8478, RMSE = R$ 507.74, MAE = R$ 404.03.

In [ ]:
from ltv_pipeline import build_final_model, evaluate_model

linear_model = build_final_model()
evaluate_model(linear_model, X_train, X_test, y_train, y_test)

## 6. Deployment

The final linear model is translated into Excel tools for business use:

- `deploy/ltv_explained_simulator.xlsx`: single-customer simulator with feature-contribution explanations.
- `deploy/ltv_operational_batch.xlsx`: operational table for scoring multiple new customers.

The spreadsheets accept the first-purchase value directly in Brazilian reais (R$).

## Limitations and next steps

The project currently uses a random 80/20 split. A production version should add temporal validation, monitor drift, recalibrate the model, quantify prediction uncertainty, and review whether demographic variables are appropriate for operational targeting.